**TE Differential Expression for SWI/SNF Loss Cohort**

This notebook repeats your TE differential expression analysis for:
ARID1A / ARID1B / SMARCA4 / SMARCB1 / PBRM1 loss

Primary model:
~ matched_set + alteration_status

Sensitivity model:
~ alteration_status + tumour_type + sex



In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(Biobase)
library(limma)
library(stringr)
library(purrr)
library(ggplot2)


setwd("/home/kennes38/ResearchProject/03_swi_snf_TEdifferentialexpression")

gene_set_label <- "swi_snf_loss"

# REdiscoverTE expression data folder
tedata_root <- "../TEdata_resources"

match_dir <- "../03_swi_snf_WT_matching"

# All differential-expression outputs go into the current directory
te_de_output_dir <- "."

case_control_file <- file.path(
  match_dir,
  paste0(gene_set_label, "_case_control_matched_table_cancer_sex_primary_tumour_only.csv")
)

file.exists(case_control_file)
dir.exists(tedata_root)
dir.exists(te_de_output_dir)

In [ ]:
# Load REdiscoverTE ExpressionSet

rds_hits <- list.files(
  tedata_root,
  pattern = "eset_TCGA_TE_intergenic_logCPM\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

stopifnot(length(rds_hits) == 1)
dat <- readRDS(rds_hits[1])

expr_full <- exprs(dat)

# Rows - TE features/subfamilies, columns are TCGA samples
# Values - REdiscoverTE-normalised log2CPM expression values
dim(expr_full)
summary(as.numeric(expr_full))

In [ ]:
# Load matched case-control table

case_control_tbl <- read_csv(case_control_file, show_col_types = FALSE)

# Table has one row per case-control pairing
# Case with two matched WT controls appears twice
case_control_tbl %>%
  summarise(
    n_cases = n_distinct(case_sample_id_16),
    n_controls = n_distinct(WT_sample_id_16),
    n_rows = n()
  )

# QC - cases and controls should both be primary tumour code 01
case_control_tbl %>%
  count(case_sample_type_code, WT_sample_type_code)

case_control_tbl %>%
  distinct(case_sample_id_16, case_project) %>%
  count(case_project, sort = TRUE)

In [ ]:
# Build analysis metadata

case_meta <- case_control_tbl %>%
  # Build one metadata row for each altered case
  transmute(
    TEdata_full_id = case_TEdata_full_id,
    sample_id_16 = case_sample_id_16,
    patient_id = case_patient_id,
    sample_type_code = case_sample_type_code,
    matched_set = case_sample_id_16,

    # 'altered' - sample has LoF and/or HomDel in at least one gene from SWI/SNF gene set
    alteration_status = "altered",
    tumour_type = case_project,
    sex = case_sex,
    genes_lost,
    events,
    lof_genes,
    homdel_genes,
    mutation_classes,
    n_genes_lost,
    n_loss_events,
    has_lof,
    has_homdel
  ) %>%
  distinct()

control_meta <- case_control_tbl %>%
  # Build one metadata row for each WT control
  transmute(
    TEdata_full_id = WT_TEdata_full_id,
    sample_id_16 = WT_sample_id_16,
    patient_id = WT_patient_id,
    sample_type_code = WT_sample_type_code,
    matched_set = case_sample_id_16,
    alteration_status = "WT",
    tumour_type = WT_project,
    sex = WT_sex,
    genes_lost = NA_character_,
    events = NA_character_,
    lof_genes = NA_character_,
    homdel_genes = NA_character_,
    mutation_classes = NA_character_,
    n_genes_lost = NA_integer_,
    n_loss_events = NA_integer_,
    has_lof = FALSE,
    has_homdel = FALSE
  ) %>%
  distinct()

analysis_meta <- bind_rows(case_meta, control_meta) %>%
  mutate(
    # WT is the reference level - limma coefficient alteration_statusaltered means altered minus WT
    alteration_status = factor(alteration_status, levels = c("WT", "altered")),
    matched_set = factor(matched_set),
    tumour_type = factor(tumour_type),
    sex = factor(sex)
  ) %>%
  distinct(TEdata_full_id, .keep_all = TRUE) %>%
  arrange(matched_set, alteration_status, TEdata_full_id)

analysis_meta %>% count(alteration_status)
analysis_meta %>% count(matched_set) %>% count(n, name = "n_matched_sets")

write_csv(
  analysis_meta,
  file.path(te_de_output_dir, paste0(gene_set_label, "_TE_analysis_sample_metadata.csv"))
)

In [ ]:
# Align expression matrix to metadata

missing_samples <- setdiff(analysis_meta$TEdata_full_id, colnames(expr_full))
missing_samples
stopifnot(length(missing_samples) == 0)

# Subset the full expression matrix to exactly the samples in the analysis
expr <- expr_full[, analysis_meta$TEdata_full_id, drop = FALSE]

# Check is important - limma assumes expression columns are in the same order as the rows of the metadata/design matrix
stopifnot(identical(colnames(expr), analysis_meta$TEdata_full_id))

dim(expr)

In [ ]:
# Load TE feature annotation

annotation_hits <- list.files(
  tedata_root,
  pattern = "repName_repClass_repFam_TE\\.RDS$",
  recursive = TRUE,
  full.names = TRUE
)

annotation_hits
stopifnot(length(annotation_hits) == 1)

annotation_raw <- readRDS(annotation_hits[1])

# REdiscoverTE annotation may store the TE feature ID as an explicit feature_id column or as row names
annotation <- annotation_raw %>%
  as.data.frame() %>%
  tibble::rownames_to_column("feature_id_from_rownames") %>%
  as_tibble()

colnames(annotation)

if (!"feature_id" %in% colnames(annotation)) {
  if (any(annotation$feature_id_from_rownames %in% rownames(expr_full))) {
    annotation <- annotation %>%
      mutate(feature_id = feature_id_from_rownames)
  } else if ("repName" %in% colnames(annotation) && any(annotation$repName %in% rownames(expr_full))) {
    annotation <- annotation %>%
      mutate(feature_id = repName)
  } else {
    stop(
      "Could not identify feature_id in the TE annotation. ",
      "Check colnames(annotation) and rownames(expr_full)."
    )
  }
}

annotation <- annotation %>%
  rename(
    TE_subfamily = repName,
    TE_class = repClass,
    TE_family = repFam
  ) %>%
  select(feature_id, TE_subfamily, TE_class, TE_family, everything())

# Check that annotation feature IDs match the expression matrix row names
cat("Annotated features overlapping expression matrix:", sum(annotation$feature_id %in% rownames(expr_full)), "\n")
cat("Expression features:", nrow(expr_full), "\n")

head(annotation)

In [ ]:
# Set limma helper function

run_limma <- function(expression_matrix, metadata, design_formula, coef_name, feature_level) {
  # Design matrix converts the metadata columns into model covariates
  design <- model.matrix(design_formula, data = metadata)

  # lmFit fits a linear model for every TE row.
  fit <- lmFit(expression_matrix, design)

  # eBayes stabilises the variance estimates across all tested TE features
  fit <- eBayes(fit, trend = TRUE)

  # topTable extracts the coefficient of interest and applies BH FDR correction
  topTable(
    fit,
    coef = coef_name,
    number = Inf,
    adjust.method = "BH",
    sort.by = "P"
  ) %>%
    rownames_to_column("feature_id") %>%
    as_tibble() %>%
    mutate(feature_level = feature_level)
}

In [ ]:
# Run matched-set model at feature/subfamily level

matched_feature_results <- run_limma(
  expression_matrix = expr,
  metadata = analysis_meta,
  design_formula = ~ matched_set + alteration_status,
  coef_name = "alteration_statusaltered",
  feature_level = "feature"
)

matched_subfamily_results <- matched_feature_results %>%
  # Add TE class/family/subfamily annotation for interpretation
  left_join(annotation, by = "feature_id") %>%
  mutate(feature_level = "subfamily")

head(matched_subfamily_results)

In [ ]:
# Aggregate expression to TE family level

expr_tbl <- as_tibble(expr, rownames = "feature_id", .name_repair = "minimal") %>%
  left_join(annotation, by = "feature_id")

family_expr_tbl <- expr_tbl %>%
  filter(!is.na(TE_family), TE_family != "") %>%
  select(-feature_id, -TE_subfamily, -TE_class) %>%
  group_by(TE_family) %>%
  summarise(across(where(is.numeric), \(x) mean(x, na.rm = TRUE)), .groups = "drop")

family_expr <- family_expr_tbl %>%
  column_to_rownames("TE_family") %>%
  as.matrix()

stopifnot(identical(colnames(family_expr), analysis_meta$TEdata_full_id))

family_results <- run_limma(
  expression_matrix = family_expr,
  metadata = analysis_meta,
  design_formula = ~ matched_set + alteration_status,
  coef_name = "alteration_statusaltered",
  feature_level = "family"
)

head(family_results)

In [ ]:
# Covariate Sensitivity model

covariate_meta <- analysis_meta %>%
  filter(
    !is.na(tumour_type),
    !is.na(sex),
    tumour_type != "",
    sex != ""
  ) %>%
  droplevels()

expr_covariate <- expr[, covariate_meta$TEdata_full_id, drop = FALSE]

stopifnot(identical(colnames(expr_covariate), covariate_meta$TEdata_full_id))

covariate_subfamily_results <- run_limma(
  expression_matrix = expr_covariate,
  metadata = covariate_meta,
  design_formula = ~ alteration_status + tumour_type + sex,
  coef_name = "alteration_statusaltered",
  feature_level = "subfamily_covariate_model"
) %>%
  left_join(annotation, by = "feature_id")

# Rebuild family-level expression for the same complete-case samples
family_expr_covariate <- family_expr[, covariate_meta$TEdata_full_id, drop = FALSE]

stopifnot(identical(colnames(family_expr_covariate), covariate_meta$TEdata_full_id))

covariate_family_results <- run_limma(
  expression_matrix = family_expr_covariate,
  metadata = covariate_meta,
  design_formula = ~ alteration_status + tumour_type + sex,
  coef_name = "alteration_statusaltered",
  feature_level = "family_covariate_model"
)

cat("Samples in matched model:", nrow(analysis_meta), "\n")
cat("Samples in covariate complete-case model:", nrow(covariate_meta), "\n")

head(covariate_subfamily_results)

In [ ]:
# ERV/LTR-only matched model

# Restrict the analysis to LTR/ERV-like TE features before running limma
erv_ltr_features <- annotation %>%
  filter(
    TE_class == "LTR" |
      str_detect(TE_family, regex("ERV|LTR", ignore_case = TRUE)) |
      str_detect(TE_subfamily, regex("ERV|LTR", ignore_case = TRUE))
  ) %>%
  pull(feature_id) %>%
  intersect(rownames(expr))

length(erv_ltr_features)

expr_erv_ltr <- expr[erv_ltr_features, , drop = FALSE]

erv_ltr_results <- run_limma(
  expression_matrix = expr_erv_ltr,
  metadata = analysis_meta,
  design_formula = ~ matched_set + alteration_status,
  coef_name = "alteration_statusaltered",
  feature_level = "ERV_LTR_subfamily"
) %>%
  left_join(annotation, by = "feature_id")

head(erv_ltr_results)

In [ ]:
# Save main results

all_results <- bind_rows(
  matched_subfamily_results,
  family_results,
  covariate_subfamily_results,
  covariate_family_results,
  erv_ltr_results
)

significance_summary <- all_results %>%
  # Nominal P < 0.05 is exploratory
  group_by(feature_level) %>%
  summarise(
    n_tested = n(),
    n_fdr_0_05 = sum(adj.P.Val < 0.05, na.rm = TRUE),
    n_fdr_0_10 = sum(adj.P.Val < 0.10, na.rm = TRUE),
    n_nominal_0_05 = sum(P.Value < 0.05, na.rm = TRUE),
    .groups = "drop"
  )

significance_summary

write_csv(matched_subfamily_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_matched_limma_results.csv")))
write_csv(family_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_family_matched_limma_results.csv")))
write_csv(covariate_subfamily_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_covariate_limma_results.csv")))
write_csv(covariate_family_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_family_covariate_limma_results.csv")))
write_csv(erv_ltr_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_ERV_LTR_matched_limma_results.csv")))
write_csv(all_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_limma_results_ALL_LEVELS.csv")))
write_csv(significance_summary, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_DE_significance_summary.csv")))

In [ ]:
# Plot candidate nominal hits by group and tumour type

# Choose nominal hits from the matched subfamily model
candidate_hits <- matched_subfamily_results %>%
  # These are exploratory hits only unless adj.P.Val also passes an FDR cutoff
  filter(P.Value < 0.05) %>%
  arrange(P.Value) %>%
  slice_head(n = 10)

candidate_hits %>%
  select(feature_id, TE_subfamily, TE_class, TE_family, logFC, P.Value, adj.P.Val)

plot_candidate <- function(candidate) {
  safe_candidate <- str_replace_all(candidate, "[^A-Za-z0-9_.-]", "_")

  plot_df <- tibble(
    TEdata_full_id = colnames(expr),
    expression = as.numeric(expr[candidate, ])
  ) %>%
    left_join(analysis_meta, by = "TEdata_full_id")

  p1 <- ggplot(plot_df, aes(x = alteration_status, y = expression, fill = alteration_status)) +
    geom_boxplot(outlier.shape = NA, alpha = 0.7) +
    geom_jitter(width = 0.15, alpha = 0.6, size = 1.5) +
    labs(
      title = candidate,
      x = NULL,
      y = "REdiscoverTE log2CPM"
    ) +
    theme_bw() +
    theme(legend.position = "none")

  p2 <- ggplot(plot_df, aes(x = interaction(tumour_type, alteration_status), y = expression, fill = alteration_status)) +
    geom_boxplot(outlier.shape = NA, alpha = 0.7) +
    geom_jitter(width = 0.15, alpha = 0.6, size = 1.2) +
    labs(
      title = paste(candidate, "by tumour type"),
      x = NULL,
      y = "REdiscoverTE log2CPM"
    ) +
    theme_bw() +
    theme(axis.text.x = element_text(angle = 90, hjust = 1))

  ggsave(file.path(te_de_output_dir, paste0(safe_candidate, "_boxplot_status.png")), p1, width = 5, height = 4, dpi = 300)
  ggsave(file.path(te_de_output_dir, paste0(safe_candidate, "_boxplot_tumour_type_status.png")), p2, width = 9, height = 5, dpi = 300)

  list(status_plot = p1, tumour_type_plot = p2)
}

if (nrow(candidate_hits) > 0) {
  plot_candidate(candidate_hits$feature_id[1])
}

In [ ]:
# Check direction consistency across tumour types

direction_by_tumour_type <- function(candidate) {
  purrr::map_dfr(split(analysis_meta, analysis_meta$tumour_type), function(meta_t) {
    samples <- meta_t$TEdata_full_id
    values <- expr[candidate, samples]

    mean_loss <- mean(values[meta_t$alteration_status == "altered"], na.rm = TRUE)
    mean_wt <- mean(values[meta_t$alteration_status == "WT"], na.rm = TRUE)

    tibble(
      feature_id = candidate,
      tumour_type = as.character(unique(meta_t$tumour_type)),
      n_loss = sum(meta_t$alteration_status == "altered"),
      n_wt = sum(meta_t$alteration_status == "WT"),
      mean_loss = mean_loss,
      mean_wt = mean_wt,
      delta_log2CPM = mean_loss - mean_wt
    )
  })
}

direction_results <- purrr::map_dfr(candidate_hits$feature_id, direction_by_tumour_type)

write_csv(
  direction_results,
  file.path(te_de_output_dir, paste0(gene_set_label, "_candidate_direction_by_tumour_type.csv"))
)

head(direction_results)

In [ ]:
# Volcano plot for matched subfamily model

volcano_df <- matched_subfamily_results %>%
  mutate(
    neg_log10_p = -log10(P.Value),
    result = case_when(
      adj.P.Val < 0.05 ~ "FDR < 0.05",
      P.Value < 0.05 ~ "Nominal P < 0.05",
      TRUE ~ "Not significant"
    )
  )

p_volcano <- ggplot(volcano_df, aes(x = logFC, y = neg_log10_p, colour = result)) +
  geom_point(alpha = 0.7, size = 1.8) +
  geom_vline(xintercept = 0, linetype = "dotted", colour = "grey60") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", colour = "grey40") +
  scale_colour_manual(
    values = c(
      "FDR < 0.05" = "#D55E00",
      "Nominal P < 0.05" = "#2C6DB2",
      "Not significant" = "grey75"
    )
  ) +
  labs(
    title = "TE subfamily differential expression",
    subtitle = "SWI/SNF loss primary tumours vs matched WT primary tumour controls",
    x = "logFC",
    y = "-log10(P value)",
    colour = "Result"
  ) +
  theme_classic()

p_volcano

ggsave(
  file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_volcano.png")),
  p_volcano,
  width = 7,
  height = 5,
  dpi = 300
)

In [ ]:
# Tumour Purity and Immune/Stromal-adjusted TE models
# Set SWI/SNF analysis directory

setwd("/home/kennes38/ResearchProject/03_swi_snf_TEdifferentialexpression")

# SWI/SNF output label used when writing adjusted-model result files
gene_set_label <- "swi_snf_loss"

te_de_output_dir <- "."

getwd()
gene_set_label
dir.exists(te_de_output_dir)

In [ ]:
# Load libraries and check required objects

user_lib <- "/home/kennes38/ResearchProject/R/x86_64-pc-linux-gnu-library/4.5"
.libPaths(c(user_lib, .libPaths()))

library(TCGAbiolinks)
library(dplyr)
library(readr)
library(tibble)
library(stringr)
library(purrr)

required_objects <- c(
  "analysis_meta",
  "expr",
  "family_expr",
  "annotation",
  "run_limma",
  "gene_set_label",
  "te_de_output_dir"
)

object_check <- tibble(
  object = required_objects,
  exists = purrr::map_lgl(required_objects, exists)
)

object_check

if (any(!object_check$exists)) {
  stop(
    "Required objects are missing. Rerun the main TE DE notebook up to the ",
    "family_expr step before running this add-on notebook."
  )
}

In [ ]:
# Set helper functions for metadata and expression alignment

add_tcga_ids <- function(df) {
  df %>%
    mutate(
      TEdata_full_id = as.character(TEdata_full_id),

      # sample_id_16 is the first 16 characters of the TCGA barcode
      sample_id_16 = substr(TEdata_full_id, 1, 16),

      sample_id_15 = substr(TEdata_full_id, 1, 15),

      # patient_id is the first 12 characters of the TCGA barcode
      patient_id = substr(TEdata_full_id, 1, 12)
    )
}

prepare_model_inputs <- function(metadata, expr_matrix, family_matrix, required_covariates) {
  model_meta <- metadata %>%
    mutate(
      TEdata_full_id = unname(as.character(TEdata_full_id))
    )

  for (covar in required_covariates) {
    model_meta <- model_meta %>%
      filter(!is.na(.data[[covar]]))

    if (is.character(model_meta[[covar]]) || is.factor(model_meta[[covar]])) {
      model_meta <- model_meta %>%
        filter(as.character(.data[[covar]]) != "")
    }
  }

  model_meta <- model_meta %>%
    distinct(TEdata_full_id, .keep_all = TRUE) %>%
    mutate(TEdata_full_id = unname(as.character(TEdata_full_id))) %>%
    droplevels()

  # Keep only samples present in the expression matrix
  model_meta <- model_meta %>%
    filter(TEdata_full_id %in% colnames(expr_matrix))

  sample_ids <- unname(as.character(model_meta$TEdata_full_id))

  model_expr <- expr_matrix[, sample_ids, drop = FALSE]
  model_family_expr <- family_matrix[, sample_ids, drop = FALSE]

  if (!identical(unname(colnames(model_expr)), sample_ids)) {
    cat("Expression/metadata mismatch detected.\n")
    cat("First expression columns:\n")
    print(head(colnames(model_expr)))
    cat("First metadata sample IDs:\n")
    print(head(sample_ids))
    stop("Expression matrix columns do not match metadata sample IDs.")
  }

  if (!identical(unname(colnames(model_family_expr)), sample_ids)) {
    stop("Family-level expression matrix columns do not match metadata sample IDs.")
  }

  list(
    metadata = model_meta,
    expr = model_expr,
    family_expr = model_family_expr
  )
}

get_erv_ltr_expr <- function(expr_matrix, annotation_tbl) {
  erv_ltr_features <- annotation_tbl %>%
    filter(
      TE_class == "LTR" |
        str_detect(TE_family, regex("ERV|LTR", ignore_case = TRUE)) |
        str_detect(TE_subfamily, regex("ERV|LTR", ignore_case = TRUE))
    ) %>%
    pull(feature_id) %>%
    intersect(rownames(expr_matrix))

  cat("ERV/LTR features retained:", length(erv_ltr_features), "\n")

  expr_matrix[erv_ltr_features, , drop = FALSE]
}

# This helper summarises how many features pass common thresholds
summarise_de_results <- function(...) {
  bind_rows(...) %>%
    group_by(feature_level) %>%
    summarise(
      n_tested = n(),
      n_fdr_0_05 = sum(adj.P.Val < 0.05, na.rm = TRUE),
      n_fdr_0_10 = sum(adj.P.Val < 0.10, na.rm = TRUE),
      n_nominal_0_05 = sum(P.Value < 0.05, na.rm = TRUE),
      .groups = "drop"
    )
}

In [ ]:
# Load TCGA tumour purity from TCGAbiolinks

# CPE -> consensus purity estimate
data("Tumor.purity")

head(Tumor.purity)
colnames(Tumor.purity)

purity_tbl <- Tumor.purity %>%
  mutate(
    purity_sample_id = as.character(Sample.ID),

    # Convert purity-table barcode to the same 16-character sample ID
    sample_id_16 = substr(purity_sample_id, 1, 16),
    patient_id = substr(purity_sample_id, 1, 12)
  ) %>%
  distinct(sample_id_16, .keep_all = TRUE)

# Add purity to your analysis metadata
analysis_meta_purity <- analysis_meta %>%
  add_tcga_ids() %>%
  left_join(
    purity_tbl %>%
      select(sample_id_16, CPE, everything()),
    by = "sample_id_16"
  ) %>%
  mutate(
    CPE = as.numeric(gsub(",", ".", as.character(CPE)))
  )

# Check how many samples have purity values
analysis_meta_purity %>%
  summarise(
    n_samples = n(),
    n_unique_samples = n_distinct(TEdata_full_id),
    n_with_CPE = sum(!is.na(CPE)),
    n_missing_CPE = sum(is.na(CPE))
  )

class(analysis_meta_purity$CPE)
summary(analysis_meta_purity$CPE)

In [ ]:
# Run tumour purity-adjusted model
purity_inputs <- prepare_model_inputs(
  metadata = analysis_meta_purity,
  expr_matrix = expr,
  family_matrix = family_expr,
  required_covariates = c("alteration_status", "tumour_type", "sex", "CPE")
)

purity_meta <- purity_inputs$metadata

expr_purity <- purity_inputs$expr

# family_expr_purity is the family-level matrix after the same sample filtering
family_expr_purity <- purity_inputs$family_expr

cat("Samples in original metadata:", nrow(analysis_meta), "\n")
cat("Samples in purity-adjusted model:", nrow(purity_meta), "\n")

purity_subfamily_results <- run_limma(
  # Rows - TE subfamilies/features, columns - samples
  expression_matrix = expr_purity,

  # Metadata rows are in the same order as expression columns
  metadata = purity_meta,

  # CPE is included as the tumour-purity covariate
  design_formula = ~ alteration_status + tumour_type + sex + CPE,
  coef_name = "alteration_statusaltered",
  feature_level = "subfamily_purity_adjusted"
) %>%
  left_join(annotation, by = "feature_id")

purity_family_results <- run_limma(
  expression_matrix = family_expr_purity,
  metadata = purity_meta,
  design_formula = ~ alteration_status + tumour_type + sex + CPE,
  coef_name = "alteration_statusaltered",
  feature_level = "family_purity_adjusted"
)

expr_erv_ltr_purity <- get_erv_ltr_expr(expr_purity, annotation)

# Same purity-adjusted model, but restricted to ERV/LTR-like features
purity_erv_ltr_results <- run_limma(
  expression_matrix = expr_erv_ltr_purity,
  metadata = purity_meta,
  design_formula = ~ alteration_status + tumour_type + sex + CPE,
  coef_name = "alteration_statusaltered",
  feature_level = "ERV_LTR_subfamily_purity_adjusted"
) %>%
  left_join(annotation, by = "feature_id")

purity_summary <- summarise_de_results(
  purity_subfamily_results,
  purity_family_results,
  purity_erv_ltr_results
)

purity_summary

write_csv(purity_subfamily_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_purity_adjusted_limma_results.csv")))
write_csv(purity_family_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_family_purity_adjusted_limma_results.csv")))
write_csv(purity_erv_ltr_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_ERV_LTR_purity_adjusted_limma_results.csv")))
write_csv(purity_summary, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_purity_adjusted_significance_summary.csv")))

In [ ]:
# Load combined ESTIMATE immune/stromal scores

estimate_file <- "../ESTIMATE_scores/combined_TCGA_ESTIMATE_scores.csv"

# This should print TRUE
file.exists(estimate_file)

estimate_tbl <- read_csv(estimate_file, show_col_types = FALSE)

colnames(estimate_tbl)
head(estimate_tbl)

required_estimate_cols <- c(
  "estimate_sample_id",
  "immune_score",
  "stromal_score",
  "estimate_score",
  "patient_id",
  "sample_id_16"
)

missing_estimate_cols <- setdiff(required_estimate_cols, colnames(estimate_tbl))

if (length(missing_estimate_cols) > 0) {
  stop(
    "The combined ESTIMATE file is missing required columns: ",
    paste(missing_estimate_cols, collapse = ", ")
  )
}

estimate_tbl <- estimate_tbl %>%
  mutate(
    # Make sure IDs and scores have the expected types
    estimate_sample_id = as.character(estimate_sample_id),
    sample_id_16 = substr(estimate_sample_id, 1, 16),

    sample_id_15 = substr(estimate_sample_id, 1, 15),

    patient_id = substr(estimate_sample_id, 1, 12),
    immune_score = as.numeric(immune_score),
    stromal_score = as.numeric(stromal_score),
    estimate_score = as.numeric(estimate_score)
  ) %>%
  # One row per sample prevents duplicate rows when joining to analysis_meta
  distinct(sample_id_16, .keep_all = TRUE)

summary(estimate_tbl$immune_score)
summary(estimate_tbl$stromal_score)

estimate_tbl %>%
  mutate(sample_type_code = substr(sample_id_16, 14, 15)) %>%
  count(sample_type_code, sort = TRUE)

In [ ]:
# Run immune/stromal-adjusted models

# Add immune and stromal scores to your analysis metadata.
analysis_meta_estimate <- analysis_meta %>%
  add_tcga_ids() %>%
  left_join(
    estimate_tbl %>%
      select(sample_id_15, immune_score, stromal_score, estimate_score, estimate_sample_id),

    by = "sample_id_15"
  )

analysis_meta_estimate %>%
  summarise(
    n_samples = n(),
    n_unique_samples = n_distinct(TEdata_full_id),
    n_with_immune = sum(!is.na(immune_score)),
    n_with_stromal = sum(!is.na(stromal_score)),
    n_missing_immune = sum(is.na(immune_score)),
    n_missing_stromal = sum(is.na(stromal_score))
  )

# This model tests:
# altered vs WT TE expression, adjusting for tumour type, sex,
# immune score and stromal score.
# In words:
# "Is the TE signal still associated with alteration status after accounting
# for immune and stromal content in the bulk tumour sample?"
estimate_inputs <- prepare_model_inputs(
  metadata = analysis_meta_estimate,
  expr_matrix = expr,
  family_matrix = family_expr,
  required_covariates = c("alteration_status", "tumour_type", "sex", "immune_score", "stromal_score")
)

estimate_meta <- estimate_inputs$metadata

# Expression matrices filtered to samples with complete immune/stromal metadata.
expr_estimate <- estimate_inputs$expr
family_expr_estimate <- estimate_inputs$family_expr

cat("Samples in original metadata:", nrow(analysis_meta), "\n")
cat("Samples in immune/stromal-adjusted model:", nrow(estimate_meta), "\n")

estimate_subfamily_results <- run_limma(
  expression_matrix = expr_estimate,
  metadata = estimate_meta,

  # immune_score and stromal_score are included as microenvironment covariates.
  design_formula = ~ alteration_status + tumour_type + sex + immune_score + stromal_score,
  coef_name = "alteration_statusaltered",
  feature_level = "subfamily_immune_stromal_adjusted"
) %>%
  left_join(annotation, by = "feature_id")

estimate_family_results <- run_limma(
  expression_matrix = family_expr_estimate,
  metadata = estimate_meta,
  design_formula = ~ alteration_status + tumour_type + sex + immune_score + stromal_score,
  coef_name = "alteration_statusaltered",
  feature_level = "family_immune_stromal_adjusted"
)

expr_erv_ltr_estimate <- get_erv_ltr_expr(expr_estimate, annotation)

# Same immune/stromal-adjusted model, but restricted to ERV/LTR-like features.
estimate_erv_ltr_results <- run_limma(
  expression_matrix = expr_erv_ltr_estimate,
  metadata = estimate_meta,
  design_formula = ~ alteration_status + tumour_type + sex + immune_score + stromal_score,
  coef_name = "alteration_statusaltered",
  feature_level = "ERV_LTR_subfamily_immune_stromal_adjusted"
) %>%
  left_join(annotation, by = "feature_id")

estimate_summary <- summarise_de_results(
  estimate_subfamily_results,
  estimate_family_results,
  estimate_erv_ltr_results
)

estimate_summary

write_csv(estimate_subfamily_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_subfamily_immune_stromal_adjusted_limma_results.csv")))
write_csv(estimate_family_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_family_immune_stromal_adjusted_limma_results.csv")))
write_csv(estimate_erv_ltr_results, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_ERV_LTR_immune_stromal_adjusted_limma_results.csv")))
write_csv(estimate_summary, file.path(te_de_output_dir, paste0(gene_set_label, "_TE_immune_stromal_adjusted_significance_summary.csv")))

In [ ]:
# Compare adjusted model summaries

adjustment_summary <- bind_rows(
  purity_summary %>%
    mutate(model = "purity_adjusted"),
  estimate_summary %>%
    mutate(model = "immune_stromal_adjusted")
) %>%
  select(model, feature_level, n_tested, n_fdr_0_05, n_fdr_0_10, n_nominal_0_05)

adjustment_summary

# This is the quickest table to send to your supervisor when asking:
# "Does the signal survive purity and immune/stromal adjustment?"
write_csv(
  adjustment_summary,
  file.path(te_de_output_dir, paste0(gene_set_label, "_TE_covariate_adjusted_model_summary.csv"))
)